In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
ip_df = pd.read_csv("../data/raw/IpAddress_to_Country.csv")

In [7]:
df = pd.read_csv("../data/raw/Fraud_Data.csv")

In [10]:
df = df.sort_values("ip_address")
ip_df = ip_df.sort_values("lower_bound_ip_address")

In [16]:
df = df.sort_values("ip_address")
ip_df = ip_df.sort_values("lower_bound_ip_address")

In [19]:
merged_df = pd.merge_asof(
    df,
    ip_df,
    left_on="ip_address",
    right_on="lower_bound_ip_address",
    direction="backward"
)

In [20]:
merged_df = merged_df[
    merged_df["ip_address"]
    <= merged_df["upper_bound_ip_address"]
]

In [22]:
merged_df["purchase_time"] = pd.to_datetime(merged_df["purchase_time"])

In [23]:
merged_df["hour_of_day"] = merged_df["purchase_time"].dt.hour

In [24]:
print(merged_df["purchase_time"].dtype)

datetime64[us]


In [25]:
merged_df["day_of_week"] = (
    merged_df["purchase_time"].dt.dayofweek
)

In [26]:
merged_df["transaction_count"] = (
    merged_df.groupby("user_id")
    ["user_id"]
    .transform("count")
)

In [27]:
merged_df = merged_df.sort_values(
    ["user_id", "purchase_time"]
)

In [28]:
merged_df["transaction_velocity"] = (
    merged_df.groupby("user_id")
    ["purchase_time"]
    .diff()
    .dt.total_seconds()
)

In [29]:
merged_df["transaction_velocity"] = (
    merged_df["transaction_velocity"]
    .fillna(0)
)

In [30]:
categorical_cols = [
    "source",
    "browser",
    "sex",
    "country"
]

In [31]:
df_encoded = pd.get_dummies(
    merged_df,
    columns=categorical_cols,
    drop_first=True
)

In [32]:
from sklearn.preprocessing import StandardScaler

In [34]:
num_cols = [
    "purchase_value",
    "age",
    "time_since_signup",
    "transaction_count",
    "transaction_velocity"
]

In [36]:
print(df_encoded.columns)
print(num_cols)

Index(['user_id', 'signup_time', 'purchase_time', 'purchase_value',
       'device_id', 'age', 'ip_address', 'class', 'lower_bound_ip_address',
       'upper_bound_ip_address',
       ...
       'country_United States', 'country_Uruguay', 'country_Uzbekistan',
       'country_Vanuatu', 'country_Venezuela', 'country_Viet Nam',
       'country_Virgin Islands (U.S.)', 'country_Yemen', 'country_Zambia',
       'country_Zimbabwe'],
      dtype='str', length=201)
['purchase_value', 'age', 'time_since_signup', 'transaction_count', 'transaction_velocity']


In [37]:
df_encoded["signup_time"] = pd.to_datetime(df_encoded["signup_time"])
df_encoded["purchase_time"] = pd.to_datetime(df_encoded["purchase_time"])

df_encoded["time_since_signup"] = (
    df_encoded["purchase_time"] - df_encoded["signup_time"]
).dt.total_seconds()

In [38]:
scaler = StandardScaler()

df_encoded[num_cols] = scaler.fit_transform(
    df_encoded[num_cols]
)

In [39]:
df_encoded.to_csv(
    "../data/processed/fraud_processed.csv",
    index=False
)